In [3]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [4]:
# Check GPU status
!nvidia-smi

Sat Mar 28 22:05:30 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090        On  | 00000000:81:00.0 Off |                  N/A |
|  0%   33C    P8              32W / 350W |     10MiB / 24576MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [5]:
# %%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install transformers==4.56.2
%pip install --no-deps trl==0.22.2

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Configuration

In [6]:
# Fix TorchDynamo bug
import os
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

In [24]:
# Model configuration
MODEL_ID = 'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit'

# Visual codebook configuration
K = 8192

BVIR_DATA_ID = 'alxxtexxr/BIVR-Data'
CODEBOOK_PATH = 'model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy'

# Model

In [8]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_ID,
    load_in_4bit = False, # False for LoRA 16bit
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
    # max_seq_length = 16384, # Must be this long for VLMs
    # fast_inference = True, # Enable vLLM fast inference
    # fast_inference = False, # Disable to fix the vLLM bug on T4
    # gpu_memory_utilization = 0.8, # Reduce if out of memory
    device_map = 'cpu', # Load in CPU first to not explode the memory usage in GPU when expanding the token embeddings
)
print(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen2_5_Vl patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.691 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionAttention(
            (qkv): Linear(in_features=1280, out_features=3840, bias=True)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
            (act_fn): SiLU()

In [10]:
# Copy the original tokenizer for comparison later
import copy

tokenizer_orig = copy.deepcopy(tokenizer)

In [11]:
list(tokenizer.tokenizer.get_added_vocab().items())[:30]

[('<|endoftext|>', 151643),
 ('<|im_start|>', 151644),
 ('<|im_end|>', 151645),
 ('<|object_ref_start|>', 151646),
 ('<|object_ref_end|>', 151647),
 ('<|box_start|>', 151648),
 ('<|box_end|>', 151649),
 ('<|quad_start|>', 151650),
 ('<|quad_end|>', 151651),
 ('<|vision_start|>', 151652),
 ('<|vision_end|>', 151653),
 ('<|vision_pad|>', 151654),
 ('<|image_pad|>', 151655),
 ('<|video_pad|>', 151656),
 ('<tool_call>', 151657),
 ('</tool_call>', 151658),
 ('<|fim_prefix|>', 151659),
 ('<|fim_middle|>', 151660),
 ('<|fim_suffix|>', 151661),
 ('<|fim_pad|>', 151662),
 ('<|repo_name|>', 151663),
 ('<|file_sep|>', 151664)]

In [12]:
def check_stats(x):
    print(f"mean: {x.mean()}, max: {x.max()}, min: {x.min()}")
    print("First values:", x[:5])
    print()

check_stats(model.language_model.embed_tokens.weight[0])
check_stats(model.language_model.embed_tokens.weight[123])
check_stats(model.language_model.embed_tokens.weight[151664])
check_stats(model.language_model.embed_tokens.weight[152063])

mean: 0.0001544952392578125, max: 0.12158203125, min: -0.1826171875
First values: tensor([-0.0171, -0.0013,  0.0189,  0.0136,  0.0270], dtype=torch.bfloat16)

mean: -0.00013637542724609375, max: 0.115234375, min: -0.0966796875
First values: tensor([-0.0231, -0.0147,  0.0280, -0.0284,  0.0181], dtype=torch.bfloat16)

mean: -2.571393892423754e-39, max: 1.1754943508222875e-37, min: -1.1754943508222875e-37
First values: tensor([ 1.1755e-37, -1.1755e-37,  1.1755e-37, -1.1755e-37, -1.1755e-37],
       dtype=torch.bfloat16)

mean: 4.1325973271096045e-39, max: 1.1754943508222875e-37, min: -1.1754943508222875e-37
First values: tensor([ 1.1755e-37, -1.1755e-37, -1.1755e-37, -1.1755e-37, -1.1755e-37],
       dtype=torch.bfloat16)



In [13]:
vtoks = [f'<|vtok_{i}|>' for i in range(K)]
tokenizer.tokenizer.add_tokens(vtoks);

In [14]:
vtok_start_id, vtok_end_id = tokenizer.tokenizer.encode(f'<|vtok_0|><|vtok_{K-1}|>')
print("Visual token start ID:", vtok_start_id)
print("Visual token end ID:", vtok_end_id)

Visual token start ID: 151665
Visual token end ID: 159856


In [15]:
model.resize_token_embeddings(vtok_end_id+1)
print(model)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionAttention(
            (qkv): Linear(in_features=1280, out_features=3840, bias=True)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
            (act_fn): SiLU()

In [21]:
model = model.to("cuda")
torch.cuda.empty_cache()

In [23]:
def check_stats(x):
    print(f"mean: {x.mean()}, max: {x.max()}, min: {x.min()}")
    print("First values:", x[:5])
    print()

check_stats(model.language_model.embed_tokens.weight[0])
check_stats(model.language_model.embed_tokens.weight[123])
check_stats(model.language_model.embed_tokens.weight[151664])
check_stats(model.language_model.embed_tokens.weight[152063])
check_stats(model.language_model.embed_tokens.weight[vtok_start_id])
check_stats(model.language_model.embed_tokens.weight[vtok_end_id])

mean: 0.0001544952392578125, max: 0.12158203125, min: -0.1826171875
First values: tensor([-0.0171, -0.0013,  0.0189,  0.0136,  0.0270], device='cuda:0',
       dtype=torch.bfloat16)

mean: -0.00013637542724609375, max: 0.115234375, min: -0.0966796875
First values: tensor([-0.0231, -0.0147,  0.0280, -0.0284,  0.0181], device='cuda:0',
       dtype=torch.bfloat16)

mean: -2.571393892423754e-39, max: 1.1754943508222875e-37, min: -1.1754943508222875e-37
First values: tensor([ 1.1755e-37, -1.1755e-37,  1.1755e-37, -1.1755e-37, -1.1755e-37],
       device='cuda:0', dtype=torch.bfloat16)

mean: 4.1325973271096045e-39, max: 1.1754943508222875e-37, min: -1.1754943508222875e-37
First values: tensor([ 1.1755e-37, -1.1755e-37, -1.1755e-37, -1.1755e-37, -1.1755e-37],
       device='cuda:0', dtype=torch.bfloat16)

mean: -9.183549615799121e-41, max: 1.1754943508222875e-37, min: -1.1754943508222875e-37
First values: tensor([-1.1755e-37, -1.1755e-37, -1.1755e-37,  1.1755e-37,  1.1755e-37],
       devic

In [25]:
from huggingface_hub import hf_hub_download

codebook_path = hf_hub_download(
    repo_id=BVIR_DATA_ID,
    filename=CODEBOOK_PATH,
    repo_type='dataset',
)
print("Visual codebook downloaded to:", codebook_path)

model_unsloth_Qwen2.5-VL-7B-Instruct-bnb(…):   0%|          | 0.00/117M [00:00<?, ?B/s]

Visual codebook downloaded to: /root/.cache/huggingface/hub/datasets--alxxtexxr--BIVR-Data/snapshots/5de7da8ec1b2cd2138e50bc11b18e17d76cbf942/model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy
